In [ ]:
import pickle
import numpy as np
import networkx as nx
import random


In [ ]:
G = nx.read_graphml(r"#### Path of the graphml file ####")

In [ ]:
edge_bc = nx.edge_betweenness_centrality(G, normalized = True, weight='weight')

In [ ]:
sorted_edges = sorted(edge_bc.items(), key=lambda x: x[1], reverse=True)

In [ ]:
largest = len(max(nx.connected_components(G), key=len))

N_SCC = len(G)
#N_WCC = len(G)
#N = G.number_of_nodes()
N = G.number_of_edges()
print(N, N_SCC)#, N_WCC)

M = int(1*N / 100)  # Number of nodes to remove

#Node_remov = np.zeros(M, int)
edge_remov = np.zeros(M, int)
L_SCC = N_SCC
#num_nodes_removed = 0
num_edges_removed = 0

edge_to_remove = []
for i in range(M):
    edge, weghts = sorted_edges[i]
    edge_to_remove.append(edge)

#Remove the edges from the graph
G.remove_edges_from(edge_to_remove)
num_edges_removed += M


# Calculate the largest connected component after edge removal
largest_cc = max(nx.connected_components(G), key=len)
L_SCC = len(largest_cc)
N_SCC = len(max(nx.connected_components(G), key=len))  # Initial largest component size
# Calculate the number of clusters (connected components)
num_clusters = len(list(nx.connected_components(G)))
print(f'Number of clusters:{num_clusters}')
# Calculate and print the ratio of removed edges and remaining component size
ratio_edges_removed = num_edges_removed / float(N)
ratio_largest_cc = L_SCC / float(N_SCC)

# Open output file to write results
with open("Size_WCC_edge_bc.dat", 'w') as fnet1:
    fnet1.write(f"{ratio_edges_removed}\t{ratio_largest_cc} \n")
    print(f"{ratio_edges_removed}\t{ratio_largest_cc}")

11654 8364
Number of clusters:18
0.009953663978033293	1.0


In [ ]:
print(f'LCC after removing the edges : {L_SCC}')
print(f'Ratio of total number of edges and LCC after removing edges = {L_SCC/N}')

LCC after removing the edges : 8307
Ratio of total number of edges and LCC after removing edges = 0.7128024712545049


In [ ]:
#G is an undirected, weighted graph
original_edge_count = G.number_of_edges()  #Total number of edges in the original graph
num_clusters_all = []
edge_bc_all = []

#Open output file
with open("Size_Lcc_navimumbai_targeted.dat", 'w') as fnet1:
    pct = 1  #Start with removing 1% each iteration
    while G.number_of_edges() > 1 and pct <= 100:
        #Calculate the number of edges to remove(1% of the original edge count)
        M = int(0.01 * original_edge_count)

        #Recalculate edge betweenness centrality for the current state of the graph
        edge_bc = nx.edge_betweenness_centrality(G, normalized=True, weight='weight')
        sorted_edges = sorted(edge_bc.items(), key=lambda x: x[1], reverse=True)

        edge_bc_all.append(edge_bc)

        #Identify top M edges to remove based on edge betweenness centrality
        edge_to_remove = []
        for i in range(min(M, len(sorted_edges))):
            edge, _ = sorted_edges[i]
            edge_to_remove.append(edge)

        #Remove the selected edges
        G.remove_edges_from(edge_to_remove)

        #Calculate the LCC after edge removal
        largest_cc_nodes = max(nx.connected_components(G), key=len)
        L_SCC = len(largest_cc_nodes)

        #Create subgraph for LCC and calculate its edge count
        largest_cc_subgraph = G.subgraph(largest_cc_nodes)
        num_edges_in_LCC = largest_cc_subgraph.number_of_edges()

        #Calculate the ratio of removed edges and LCC edge ratio
        ratio_edges_removed = (original_edge_count - G.number_of_edges()) / float(original_edge_count)
        ratio_largest_cc = num_edges_in_LCC / float(original_edge_count)

        #Calculate the number of clusters (connected components)
        num_clusters = len(list(nx.connected_components(G)))
        print(f'Largest Connected component (nodes): {L_SCC}')
        print(f'Number of edges in LCC: {num_edges_in_LCC}')
        print(f'Number of Clusters: {num_clusters}')
        num_clusters_all.append(num_clusters)

        #Write results to file and print them
        fnet1.write(f"{pct}\t{ratio_edges_removed}\t{ratio_largest_cc}\t{num_clusters}\n")
        print(f"{pct}% edges removed cumulatively: {ratio_edges_removed:.6f}, Largest CC edge ratio: {ratio_largest_cc:.6f}, Clusters: {num_clusters}")

        #Increment percentage step (even though we're removing top 1% each iteration, this helps to log cumulative impact)
        pct += 1

        #Stop if the graph has no more edges
        if G.number_of_edges() == 0:
            break


Largest Connected component (nodes): 8233
Number of edges in LCC: 11329
Number of Clusters: 45
1% edges removed cumulatively: 0.009967, Largest CC edge ratio: 0.981886, Clusters: 45
Largest Connected component (nodes): 8161
Number of edges in LCC: 11156
Number of Clusters: 63
2% edges removed cumulatively: 0.019934, Largest CC edge ratio: 0.966892, Clusters: 63
Largest Connected component (nodes): 6508
Number of edges in LCC: 8891
Number of Clusters: 94
3% edges removed cumulatively: 0.029901, Largest CC edge ratio: 0.770584, Clusters: 94
Largest Connected component (nodes): 2759
Number of edges in LCC: 3893
Number of Clusters: 123
4% edges removed cumulatively: 0.039868, Largest CC edge ratio: 0.337407, Clusters: 123
Largest Connected component (nodes): 2597
Number of edges in LCC: 3655
Number of Clusters: 156
5% edges removed cumulatively: 0.049835, Largest CC edge ratio: 0.316779, Clusters: 156
Largest Connected component (nodes): 2139
Number of edges in LCC: 3031
Number of Clusters